# MP2 Part 1 — commit retrieval (melawady)
Retrieves every commit for my 10 assigned projects from World of Code, plus stars / forks / last commit date from GitHub.

Commit contents are cached per project in `cache/*.jsonl`, so if the run is interrupted (wandb_client and pytorch_audio are large) just re-run the cells and it resumes where it stopped.

In [ ]:
%pip install -U pandas matplotlib requests tqdm
# python-woc is not used: it only ships Linux builds and fails to compile on Windows
# please clear the output of this cell in your notebook before checking it in

Note: you may need to restart the kernel to use updated packages.


In [ ]:
NETID = 'melawady'
PROJECTS = {  # WoC project id -> GitHub repo (from net2prj.csv, netid melawady)
    'nickabattista_ib2d':      'https://github.com/nickabattista/IB2d',
    'cartavis_carta-frontend': 'https://github.com/CARTAvis/carta-frontend',
    'nanocomp_mpb':            'https://github.com/NanoComp/mpb',
    'kklmn_xrt':               'https://github.com/kklmn/xrt',
    'pytorch_audio':           'https://github.com/pytorch/audio',
    'bernatgel_karyoploter':   'https://github.com/bernatgel/karyoploteR',
    'wandb_client':            'https://github.com/wandb/client',
    'ohdsi_atlas':             'https://github.com/OHDSI/Atlas',
    'mummer4_mummer':          'https://github.com/mummer4/mummer',
    'nmslib_nmslib':           'https://github.com/nmslib/nmslib',
}

# Optional: a GitHub personal access token (no scopes needed) raises the GitHub API limit
# from 60 to 5000 requests/hour. Leave empty to go unauthenticated. Don't commit a real token.
GH_TOKEN = ''

## 1. Commit sha1s per project (WoC `p2c` map)

In [ ]:
import os, time, json
import pandas as pd
from tqdm import tqdm
import requests

class WocRemote:
    """Minimal stand-in for woc.remote.WocMapsRemote (same REST calls, same return values).
    python-woc has no Windows builds and needs a C++ compiler to install, so we call the API directly."""
    def __init__(self, base_url='https://worldofcode.org/api', api_key=None):
        self.base_url = base_url.rstrip('/')
        self.s = requests.Session()
        if api_key:
            self.s.headers['Authorization'] = f'Bearer {api_key}'

    def _get(self, path, params=None):
        r = self.s.get(self.base_url + path, params=params, timeout=(30, 180))
        if r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 60))
            time.sleep(wait)  # server asked us to back off
            raise RuntimeError(f'rate limited, waited {wait}s')
        r.raise_for_status()
        return r.json()

    def get_values(self, map_name, key):
        return self._get(f'/lookup/map/{map_name}/{key}')['data']

    def get_values_many(self, map_name, keys):
        j = self._get(f'/lookup/map/{map_name}', params=[('q', k) for k in keys])
        return j['data'], j.get('errors', {})

woc = WocRemote()
# woc = WocRemote(api_key="woc-XXXXXX-YYYYYY")

if os.path.exists('df_commits.csv'):
    df_shas = pd.read_csv('df_commits.csv')
else:
    rows = []
    for prj in PROJECTS:
        shas = woc.get_values('p2c', prj)
        print(f'{prj:26s} {len(shas):6d} commit sha1s')
        rows += [(prj, s) for s in shas]
        time.sleep(1)
    df_shas = pd.DataFrame(rows, columns=['project', 'sha1']).drop_duplicates()
    df_shas.to_csv('df_commits.csv', index=False)

df_shas.groupby('project').size().rename('sha1s')

project
bernatgel_karyoploter        674
cartavis_carta-frontend    12488
kklmn_xrt                   1592
mummer4_mummer               413
nanocomp_mpb                1351
nickabattista_ib2d           994
nmslib_nmslib               1995
ohdsi_atlas                 6665
pytorch_audio              15864
wandb_client               54754
Name: sha1s, dtype: int64

## 2. Commit contents (WoC `commit.tch`), batches of 10 with a 1 s wait between requests

In [ ]:
BATCH = 10   # README: keep chunks well under 50 and wait between requests
os.makedirs('cache', exist_ok=True)

def fetch_batch(batch, tries=25):
    # WoC is often overloaded (everyone in the class hits it at once): keep retrying for a long time
    for attempt in range(tries):
        try:
            res, err = woc.get_values_many('commit.tch', batch)
            return {k: v[0] for k, v in res.items()}, err
        except Exception as e:
            wait = min(300, 10 * 2 ** attempt)
            print(f'  request failed ({type(e).__name__}: {str(e)[:120]}); retry {attempt + 1}/{tries} in {wait}s')
            time.sleep(wait)
    raise RuntimeError('giving up on a batch - re-run this cell later to resume')

for prj, grp in df_shas.groupby('project'):
    path = f'cache/{prj}.jsonl'
    done = set()
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            done = {json.loads(line)['commit'] for line in f}
    todo = [s for s in grp['sha1'] if s not in done]
    print(f'{prj}: {len(done)} cached, {len(todo)} to fetch')
    with open(path, 'a', encoding='utf-8') as out:
        for i in tqdm(range(0, len(todo), BATCH), desc=prj):
            res, err = fetch_batch(todo[i:i + BATCH])
            if err:
                print('  WoC reported errors:', err)
            for sha, c in res.items():
                # c = [tree, [parents], [author, unixtime, tz], [committer, unixtime, tz], message]
                out.write(json.dumps({'project': prj, 'commit': sha, 'author': c[2][0],
                                      'time': int(c[2][1]), 'message': c[4]}) + '\n')
            out.flush()
            time.sleep(1)

bernatgel_karyoploter: 674 cached, 0 to fetch


bernatgel_karyoploter: 0it [00:00, ?it/s]


cartavis_carta-frontend: 740 cached, 11748 to fetch


cartavis_carta-frontend:   9%|▉         | 106/1175 [01:49<18:35,  1.04s/it]

  WoC reported errors: {'24a6bc650d2911ac890056402dc90f9745d995b9': 'Key 24a6bc650d2911ac890056402dc90f9745d995b9 not found in /da5_fast/All.sha1c/commit_36.tch'}


...

wandb_client:  81%|████████▏ | 4459/5476 [1:16:52<17:29,  1.03s/it]

  WoC reported errors: {'d0b437efc86a38c39858db8b2f5b0aa34be00412': 'Key d0b437efc86a38c39858db8b2f5b0aa34be00412 not found in /da5_fast/All.sha1c/commit_80.tch'}


wandb_client:  89%|████████▊ | 4859/5476 [1:23:45<10:36,  1.03s/it]

  WoC reported errors: {'e38f2ed3b43185617dd1c49bf24d51a6f6cce0db': 'Key e38f2ed3b43185617dd1c49bf24d51a6f6cce0db not found in /da5_fast/All.sha1c/commit_99.tch'}


wandb_client: 100%|██████████| 5476/5476 [1:34:23<00:00,  1.03s/it]


## 3. Write `melawady_project_summary.csv` (semicolon separated) and validate counts

In [ ]:
recs = []
for prj in PROJECTS:
    with open(f'cache/{prj}.jsonl', encoding='utf-8') as f:
        recs += [json.loads(line) for line in f]

summary = (pd.DataFrame(recs)
             .drop_duplicates(['project', 'commit'])
             .rename(columns={'project': 'project_wocid', 'commit': 'commit_sha1', 'message': 'commit message'})
             [['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']]
             .sort_values(['project_wocid', 'time']))
summary.to_csv(f'{NETID}_project_summary.csv', sep=';', index=False)

# validation: every sha1 from p2c should have content
check = (df_shas.groupby('project').size().rename('p2c_sha1s').to_frame()
         .join(summary.groupby('project_wocid').size().rename('retrieved')))
check['missing'] = check['p2c_sha1s'] - check['retrieved']
check

,p2c_sha1s,retrieved,missing
project,,,
bernatgel_karyoploter,674,674,0
cartavis_carta-frontend,12488,12474,14
kklmn_xrt,1592,1591,1
mummer4_mummer,413,405,8
nanocomp_mpb,1351,1335,16
nickabattista_ib2d,994,994,0
nmslib_nmslib,1995,1995,0
ohdsi_atlas,6665,6636,29
pytorch_audio,15864,15856,8


## 4. WoC statistics per project

In [ ]:
woc_stats = summary.groupby('project_wocid').agg(
    ncommits=('commit_sha1', 'nunique'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max'))
woc_stats['min_date'] = pd.to_datetime(woc_stats['min_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats['max_date'] = pd.to_datetime(woc_stats['max_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats.to_csv(f'{NETID}_woc_stats.csv', sep=';')
woc_stats

,ncommits,nauthors,min_time,max_time,min_date,max_date
project_wocid,,,,,,
bernatgel_karyoploter,674,15,1444405906,1749198667,2015-10-09,2025-06-06
cartavis_carta-frontend,12474,76,1524835993,1762270305,2018-04-27,2025-11-04
kklmn_xrt,1591,12,1459266517,1762357339,2016-03-29,2025-11-05
mummer4_mummer,405,29,1352343979,1739151225,2012-11-08,2025-02-10
nanocomp_mpb,1335,35,895645130,1775155248,1998-05-20,2026-04-02
nickabattista_ib2d,994,28,1434499605,1756008042,2015-06-17,2025-08-24
nmslib_nmslib,1995,100,1373454367,1776120170,2013-07-10,2026-04-13
ohdsi_atlas,6636,139,1436372795,1761750248,2015-07-08,2025-10-29
pytorch_audio,15856,1513,1493944663,1776253805,2017-05-05,2026-04-15


## 5. GitHub statistics (stars, forks, last commit date on the default branch)
Also saves the 10 most recent GitHub commits per project, used for `RecentThemes` in Part 3.

In [ ]:
import requests

H = {'Accept': 'application/vnd.github+json'}
if GH_TOKEN:
    H['Authorization'] = f'Bearer {GH_TOKEN}'

def gh(url):
    r = requests.get(url, headers=H, timeout=30)
    r.raise_for_status()
    return r.json()

gh_rows, recent = [], []
for prj, url in PROJECTS.items():
    full = url.rstrip('/').split('github.com/')[1]
    try:
        repo = gh(f'https://api.github.com/repos/{full}')          # follows renamed/transferred repos
        commits = gh(f"https://api.github.com/repos/{repo['full_name']}/commits"
                     f"?sha={repo['default_branch']}&per_page=10")
        gh_rows.append({'project': prj, 'gh_full_name': repo['full_name'],
                        'nstars': repo['stargazers_count'], 'nforks': repo['forks_count'],
                        'lastGHCommitDate': commits[0]['commit']['committer']['date'][:10],
                        'archived': repo['archived']})
        for c in commits:
            recent.append({'project': prj, 'commit_sha1': c['sha'],
                           'author': f"{c['commit']['author']['name']} <{c['commit']['author']['email']}>",
                           'date': c['commit']['author']['date'], 'commit message': c['commit']['message']})
    except Exception as e:
        print(f'{prj}: GitHub lookup failed ({e}). Check {url} in a browser and fill in by hand.')
        gh_rows.append({'project': prj})
    time.sleep(1)

gh_stats = pd.DataFrame(gh_rows).set_index('project')
gh_stats.to_csv(f'{NETID}_gh_stats.csv', sep=';')
pd.DataFrame(recent).to_csv(f'{NETID}_gh_recent_commits.csv', sep=';', index=False)
gh_stats

,gh_full_name,nstars,nforks,lastGHCommitDate,archived
project,,,,,
nickabattista_ib2d,nickabattista/IB2d,204,101,2026-06-14,False
cartavis_carta-frontend,CARTAvis/carta-frontend,19,15,2026-09-23,False
nanocomp_mpb,NanoComp/mpb,214,107,2026-05-27,False
kklmn_xrt,kklmn/xrt,101,39,2026-09-24,False
pytorch_audio,pytorch/audio,2945,799,2026-09-23,False
bernatgel_karyoploter,bernatgel/karyoploteR,377,43,2026-04-28,False
wandb_client,wandb/wandb,11259,899,2026-09-25,False
ohdsi_atlas,OHDSI/Atlas,323,154,2025-11-09,False
mummer4_mummer,mummer4/mummer,575,119,2025-02-04,False


## 6. Tables for the text cells below
Run this, then paste its output into the two markdown cells that follow.

In [ ]:
def md_table(df):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    lines += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in df.itertuples(index=False)]
    return '\n'.join(lines)

print(md_table(gh_stats.reset_index()[['project', 'nstars', 'nforks', 'lastGHCommitDate']]))
print()
print(md_table(woc_stats.reset_index()[['project_wocid', 'ncommits', 'nauthors', 'min_time', 'min_date', 'max_time', 'max_date']]))

| project | nstars | nforks | lastGHCommitDate |
|---|---|---|---|
| nickabattista_ib2d | 204 | 101 | 2026-06-14 |
| cartavis_carta-frontend | 19 | 15 | 2026-09-23 |
| nanocomp_mpb | 214 | 107 | 2026-05-27 |
| kklmn_xrt | 101 | 39 | 2026-09-24 |
| pytorch_audio | 2945 | 799 | 2026-09-23 |
| bernatgel_karyoploter | 377 | 43 | 2026-04-28 |
| wandb_client | 11259 | 899 | 2026-09-25 |
| ohdsi_atlas | 323 | 154 | 2025-11-09 |
| mummer4_mummer | 575 | 119 | 2025-02-04 |
| nmslib_nmslib | 3591 | 461 | 2026-01-12 |

| project_wocid | ncommits | nauthors | min_time | min_date | max_time | max_date |
|---|---|---|---|---|---|---|
| bernatgel_karyoploter | 674 | 15 | 1444405906 | 2015-10-09 | 1749198667 | 2025-06-06 |
| cartavis_carta-frontend | 12474 | 76 | 1524835993 | 2018-04-27 | 1762270305 | 2025-11-04 |
| kklmn_xrt | 1591 | 12 | 1459266517 | 2016-03-29 | 1762357339 | 2025-11-05 |
| mummer4_mummer | 405 | 29 | 1352343979 | 2012-11-08 | 1739151225 | 2025-02-10 |
| nanocomp_mpb | 1335 | 35 | 8

### GitHub info (stars, forks, last commit date)

_Paste the first table printed above here._

### World of Code info (commits, authors, min time, max time)

_Paste the second table printed above here._

Note: WoC data runs to about Jan 2025, so projects with later GitHub activity will show fewer commits in WoC than on GitHub.

# Make sure you check in to your fork not just the notebook but also the csv files!!!